In [37]:
import sys; sys.path.insert(0, '..')

# Ensure local utils/ is found before labs/02/utils/
import importlib
if 'utils' in sys.modules:
    del sys.modules['utils']
sys.path.insert(0, '.')

import ugradiolab.plotting as plotting  # applies rcParams
from utils import (flag_rfi_channels, flag_outlier_dumps, vlsr_correction,
                   compute_R_for_dumps, compute_cell_metrics, neighbor_qa)

import re
from pathlib import Path
import datetime as dt
import numpy as np

SAMPLE_RATE_HZ = 2.56e6
NFFT = 1024
HI_REST_MHZ = 1420.405
C_KMS = 299792.458

%matplotlib inline

## 1. Load all scan dumps

In [38]:
STREAMING_DIR = Path('../../../data/lab04/streaming')

scan_dirs = []
for session_dir in sorted(STREAMING_DIR.glob('session_*')):
    found = sorted(session_dir.glob('obs_*'))
    print(f'{session_dir.name}: {len(found)} obs cells')
    scan_dirs.extend(found)

print(f'Total: {len(scan_dirs)} obs cells')

# Collect all dumps
records = []
for d in scan_dirs:
    session_label = d.parent.name
    for p in sorted(d.glob('*.npz')):
        with np.load(p, allow_pickle=True) as f:
            records.append({
                'path': p,
                'session': session_label,
                'target': str(f['target_name']),
                'corr00': f['corr00'].astype(float),
                'corr11': f['corr11'].astype(float),
                'lo_mhz': float(f['lo_freq_mhz']),
                'noise_on': bool(f['noise_on']),
                'time': float(f['time']),
                'alt': float(f['alt_deg']),
                'az': float(f['az_deg']),
                'ra': float(f['ra_deg']),
                'dec': float(f['dec_deg']),
            })

N = len(records)
print(f'{N} total dumps loaded')

# Extract galactic coordinates directly from target_name (obs_{l}_{b} format)
for r in records:
    m = re.match(r'(?:obs|cal)_(-?\d+)_(-?\d+)', r['target'])
    if m:
        r['gl'] = int(m.group(1))
        r['gb'] = int(m.group(2))
    else:
        r['gl'] = None
        r['gb'] = None

sessions = sorted(set(r['session'] for r in records))
lo_unique = sorted(set(r['lo_mhz'] for r in records if not r['noise_on']))
n_cal = sum(1 for r in records if r['noise_on'])
n_sci = sum(1 for r in records if not r['noise_on'])

for s in sessions:
    n_s = sum(1 for r in records if r['session'] == s)
    if n_s > 0:
        print(f'  {s}: {n_s} dumps')
gl_vals = sorted(set(r['gl'] for r in records if r['gl'] is not None))
gb_vals = sorted(set(r['gb'] for r in records if r['gb'] is not None))
print(f'Grid: {len(gb_vals)} b-rows x {len(gl_vals)} l-cols')
print(f'LO: {lo_unique}')
print(f'Cal dumps: {n_cal}, Science dumps: {n_sci}')


session_001: 153 obs cells
session_002: 14 obs cells
session_003: 95 obs cells
session_004: 67 obs cells
session_005: 19 obs cells
session_006: 97 obs cells
session_007: 80 obs cells
session_008: 75 obs cells
session_009: 153 obs cells
session_010: 40 obs cells
session_011: 32 obs cells
session_012: 241 obs cells
session_013: 17 obs cells
session_014: 77 obs cells
session_015: 7 obs cells
session_016: 26 obs cells
session_017: 4 obs cells
session_018: 101 obs cells
session_019: 4 obs cells
session_020: 1 obs cells
session_021: 17 obs cells
session_022: 4 obs cells
session_023: 15 obs cells
session_024: 3 obs cells
session_025: 3 obs cells
session_026: 51 obs cells
session_027: 169 obs cells
session_029: 2 obs cells
session_030: 7 obs cells
session_031: 15 obs cells
session_032: 5 obs cells
session_033: 3 obs cells
session_034: 33 obs cells
session_035: 38 obs cells
session_036: 14 obs cells
session_037: 21 obs cells
Total: 1703 obs cells
13792 total dumps loaded
  session_001: 1221 dum

## 2. fftshift + mask DC + RFI flagging

In [39]:
DC_BIN = NFFT // 2
f_bb_mhz = np.fft.fftshift(np.fft.fftfreq(NFFT, d=1.0/SAMPLE_RATE_HZ)) / 1e6
df_khz = SAMPLE_RATE_HZ / NFFT / 1e3

for r in records:
    r['corr00'] = np.fft.fftshift(r['corr00'])
    r['corr11'] = np.fft.fftshift(r['corr11'])

    # Mask DC bin (LO spike at baseband 0 = channel 512 after fftshift)
    r['corr00'][DC_BIN] = np.nan
    r['corr11'][DC_BIN] = np.nan

    # For LO=1420: mask 2 bins below DC to suppress leakage near HI line.
    if r['lo_mhz'] == 1420.0:
        for ch in [DC_BIN - 2, DC_BIN - 1]:
            r['corr00'][ch] = np.nan
            r['corr11'][ch] = np.nan

    r['stokes_I'] = r['corr00'] + r['corr11']

# RFI flagging: rolling median + MAD, flags both positive and
# negative outliers (spikes and dropouts).
RFI_WINDOW = 15
RFI_SIGMA = 5.0
n_flagged = sum(flag_rfi_channels(r['stokes_I'], RFI_WINDOW, RFI_SIGMA)
                for r in records)

print(f'Baseband: [{f_bb_mhz[0]:.3f}, {f_bb_mhz[-1]:.3f}] MHz')
print(f'dnu = {df_khz:.2f} kHz')
print(f'Masked DC bin: channel {DC_BIN}')
print(f'Masked DC leakage: channels {DC_BIN-2}, {DC_BIN-1} (LO=1420 only)')
print(f'RFI flagged: {n_flagged} samples across {N} dumps')

Baseband: [-1.280, 1.278] MHz
dnu = 2.50 kHz
Masked DC bin: channel 512
Masked DC leakage: channels 510, 511 (LO=1420 only)
RFI flagged: 1186627 samples across 13792 dumps


In [40]:
import warnings

SHAPE_DEV_THRESH = 0.10
SHAPE_FRAC_THRESH = 0.20

outlier_records = flag_outlier_dumps(records, SHAPE_DEV_THRESH, SHAPE_FRAC_THRESH)
n_outlier_removed = len(outlier_records)
N = len(records)

print(f'Outlier dump filter (spectral shape): removed {n_outlier_removed} dumps')
print(f'  (flag if >{SHAPE_FRAC_THRESH:.0%} of channels deviate '
      f'>{SHAPE_DEV_THRESH:.0%} from group median)')
print(f'Remaining: {N} dumps')

Outlier dump filter (spectral shape): removed 27 dumps
  (flag if >20% of channels deviate >10% from group median)
Remaining: 13765 dumps


## 3. Frequency-switched profiles per cell

For each cell, pair dumps at the two LO frequencies and compute
`R = (I1 - I2) / I2`.

**LSR correction**: Each dump's topocentric velocity is shifted to the
kinematic LSR frame (solar motion 20 km/s toward 18h +30deg B1900).
Combined results interpolate each session onto a common LSR velocity grid
before averaging, correcting for Earth's orbital motion between sessions.

In [41]:
import astropy.coordinates as ac
import astropy.units as u_ast
from astropy.time import Time as AstroTime
from collections import defaultdict

EDGE_TRIM_MHZ = 0.256
MOLL_CENTER_L = 120.0

lo1, lo2 = lo_unique[0], lo_unique[1]
f_sky = lo1 + f_bb_mhz
f_sky_2 = lo2 + f_bb_mhz
f_overlap_lo = max(f_sky[0], f_sky_2[0]) + EDGE_TRIM_MHZ
f_overlap_hi = min(f_sky[-1], f_sky_2[-1]) - EDGE_TRIM_MHZ
overlap_mask = (f_sky >= f_overlap_lo) & (f_sky <= f_overlap_hi)
f_overlap = f_sky[overlap_mask]
v_overlap = C_KMS * (1 - f_overlap / HI_REST_MHZ)  # topocentric
dv_kms = np.abs(np.median(np.diff(v_overlap)))

print(f'LO pair: ({lo1}, {lo2}) MHz')
print(f'Overlap: [{f_overlap_lo:.2f}, {f_overlap_hi:.2f}] MHz')
print(f'Velocity (topo): [{v_overlap[-1]:.0f}, {v_overlap[0]:.0f}] km/s')
print(f'dv = {dv_kms:.3f} km/s per channel')

# --- Assign galactic coords ---
for r in records:
    if r.get('gl') is None:
        r['gl'], r['gb'] = None, None
        continue
    c = ac.SkyCoord(ra=r['ra'] * u_ast.deg, dec=r['dec'] * u_ast.deg, frame='icrs')
    r['gl'] = round(c.galactic.l.deg)
    r['gb'] = round(c.galactic.b.deg)

# --- LSR velocity correction ---
# v_LSR = v_topo + v_corr, where v_corr = heliocentric + solar motion to LSRK.
# Compute one correction per (DR, cell) group -- all dumps in a group are
# taken within minutes, so share the same correction to < 0.01 km/s.
print('Computing LSR corrections...')
cell_dr_groups = defaultdict(list)
for r in records:
    if r.get('gl') is None or r['noise_on']:
        r['v_corr_lsr'] = 0.0
        continue
    cell_dr_groups[(r['session'], r['gl'], r['gb'])].append(r)

for key, group in cell_dr_groups.items():
    r0 = group[0]
    mean_t = np.mean([r['time'] for r in group])
    v_corr = vlsr_correction(r0['ra'], r0['dec'], mean_t)
    for r in group:
        r['v_corr_lsr'] = v_corr

sci_vcorr = [r['v_corr_lsr'] for r in records
             if r.get('gl') is not None and not r['noise_on']]
mean_vcorr = np.mean(sci_vcorr)
print(f'  Range: {min(sci_vcorr):.2f} to {max(sci_vcorr):.2f} km/s '
      f'(mean {mean_vcorr:.2f})')

# Common LSR velocity grid for combined results
v_lsr_overlap = v_overlap + mean_vcorr
print(f'Velocity (LSR):  [{v_lsr_overlap[-1]:.0f}, {v_lsr_overlap[0]:.0f}] km/s')

# Per-DR LSR corrections
for dr in sessions:
    dr_vc = [r['v_corr_lsr'] for r in records
             if r['session'] == dr and r.get('gl') is not None and not r['noise_on']]
    if dr_vc:
        print(f'  {dr}: v_corr = {np.mean(dr_vc):+.2f} km/s '
              f'(shift vs ref: {np.mean(dr_vc) - mean_vcorr:+.2f})')

# --- All unique (l, b) pointings ---
all_pointings = sorted(set((r['gl'], r['gb']) for r in records
                           if r['gl'] is not None))

# --- Build results ---
cell_results = {}           # per-DR, topocentric (for per-DR plots)
cell_results_combined = {}  # combined, LSR-corrected

for gl, gb in all_pointings:
    sci_dumps = [r for r in records
                 if r['gl'] == gl and r['gb'] == gb and not r['noise_on']]

    # Combined with LSR correction
    result = compute_R_for_dumps(sci_dumps, lo1, lo2, overlap_mask, v_overlap,
                                 lsr_correct=True, v_lsr_grid=v_lsr_overlap)
    if result is not None:
        cell_results_combined[(gl, gb)] = result

    # Per DR (topocentric -- intra-session shift is negligible)
    for dr in sessions:
        dr_dumps = [r for r in sci_dumps if r['session'] == dr]
        result = compute_R_for_dumps(dr_dumps, lo1, lo2, overlap_mask, v_overlap)
        if result is not None:
            cell_results[(dr, gl, gb)] = result

for dr in sessions:
    n = sum(1 for k in cell_results if k[0] == dr)
    print(f'  {dr}: {n} cells')
print(f'  Combined (LSR): {len(cell_results_combined)} cells')

LO pair: (1420.0, 1421.0) MHz
Overlap: [1419.98, 1421.02] MHz
Velocity (topo): [-130, 90] km/s
dv = 0.528 km/s per channel
Computing LSR corrections...
  Range: -43.89 to 43.33 km/s (mean -14.04)
Velocity (LSR):  [-144, 76] km/s
  session_001: v_corr = +24.45 km/s (shift vs ref: +38.49)
  session_002: v_corr = -33.58 km/s (shift vs ref: -19.54)
  session_003: v_corr = +11.25 km/s (shift vs ref: +25.29)
  session_004: v_corr = -32.89 km/s (shift vs ref: -18.85)
  session_005: v_corr = -39.99 km/s (shift vs ref: -25.95)
  session_006: v_corr = -36.02 km/s (shift vs ref: -21.98)
  session_007: v_corr = +28.55 km/s (shift vs ref: +42.59)
  session_008: v_corr = -18.46 km/s (shift vs ref: -4.42)
  session_009: v_corr = -36.79 km/s (shift vs ref: -22.75)
  session_010: v_corr = -41.64 km/s (shift vs ref: -27.60)
  session_011: v_corr = -3.51 km/s (shift vs ref: +10.53)
  session_012: v_corr = -37.16 km/s (shift vs ref: -23.12)
  session_013: v_corr = +40.71 km/s (shift vs ref: +54.76)
  sess

/home/ikaros/projects/ay-121/labs/04/notebooks/utils/freqswitch.py:70: RuntimeWarning: Mean of empty slice
  R_sess = np.nanmean(R_pairs, axis=0)
/home/ikaros/projects/ay-121/labs/04/notebooks/utils/freqswitch.py:95: RuntimeWarning: Mean of empty slice
  R_mean = np.nanmean(R_all, axis=0)
/home/ikaros/projects/ay-121/labs/04/notebooks/utils/freqswitch.py:107: RuntimeWarning: Mean of empty slice
  R_mean = np.nanmean(R_cat, axis=0)


  session_001: 153 cells
  session_002: 14 cells
  session_003: 95 cells
  session_004: 67 cells
  session_005: 19 cells
  session_006: 97 cells
  session_007: 80 cells
  session_008: 75 cells
  session_009: 153 cells
  session_010: 40 cells
  session_011: 32 cells
  session_012: 241 cells
  session_013: 17 cells
  session_014: 76 cells
  session_015: 7 cells
  session_016: 26 cells
  session_017: 4 cells
  session_018: 100 cells
  session_019: 4 cells
  session_020: 1 cells
  session_021: 16 cells
  session_022: 4 cells
  session_023: 14 cells
  session_024: 3 cells
  session_025: 3 cells
  session_026: 51 cells
  session_027: 169 cells
  session_029: 2 cells
  session_030: 7 cells
  session_031: 15 cells
  session_032: 5 cells
  session_033: 3 cells
  session_034: 33 cells
  session_035: 38 cells
  session_036: 13 cells
  session_037: 21 cells
  Combined (LSR): 1636 cells


## 4. Neighbor-based QA

Flag cells whose integrated intensity or peak velocity deviates
from the beam-weighted local plane fit of their neighbors.

Uses `utils.qa.compute_cell_metrics` and `utils.qa.neighbor_qa`.

In [42]:
cell_metrics = compute_cell_metrics(cell_results_combined, v_lsr_overlap, dv_kms)
neighbor_cells = neighbor_qa(cell_metrics, dv_kms=dv_kms)

neighbor_flagged = [c for c in neighbor_cells if c['W_flag'] or c['peak_v_flag']]
neighbor_w_flag_count = sum(1 for c in neighbor_cells if c['W_flag'])
neighbor_peak_flag_count = sum(1 for c in neighbor_cells if c['peak_v_flag'])

print(f'Neighbor QA: {len(neighbor_cells)} cells analyzed')
print(f'  Flags: W={neighbor_w_flag_count}, peak_v={neighbor_peak_flag_count}, '
      f'any={len(neighbor_flagged)}')

if neighbor_flagged:
    print('  Most deviant cells:')

    def severity(cell):
        w_score = abs(cell['W_frac_resid']) if np.isfinite(cell['W_frac_resid']) else 0.0
        v_score = abs(cell['peak_v_z']) if np.isfinite(cell['peak_v_z']) else 0.0
        return max(w_score, v_score)

    for cell in sorted(neighbor_flagged, key=severity, reverse=True)[:12]:
        print(
            f"    l={cell['gl']:3d} b={cell['gb']:3d} "
            f"W={cell['W']:+.3f} (frac={cell['W_frac_resid']:+.2f}, z={cell['W_z']:+.2f}) "
            f"v_peak={cell['peak_v']:+.1f} km/s "
            f"(dv={cell['peak_v_resid']:+.1f}, z={cell['peak_v_z']:+.2f}) "
            f"n={cell['neighbor_count']}"
        )

Neighbor QA: 1636 cells analyzed
  Flags: W=16, peak_v=4, any=20
  Most deviant cells:
    l=208 b=  0 W=+35.268 (frac=+0.05, z=+0.43) v_peak=+41.9 km/s (dv=+23.8, z=+6.32) n=36
    l=128 b= 16 W=+10.587 (frac=+0.07, z=+5.50) v_peak=-11.9 km/s (dv=-15.5, z=-5.15) n=6
    l=115 b= -1 W=+36.276 (frac=+0.05, z=+1.40) v_peak=-46.8 km/s (dv=-15.2, z=-5.06) n=54
    l=123 b=  1 W=+37.473 (frac=+0.19, z=+3.52) v_peak=-54.7 km/s (dv=-40.3, z=-4.19) n=34
    l= 12 b= -2 W=+16.085 (frac=-0.34, z=-5.27) v_peak=+9.2 km/s (dv=-8.8, z=-2.95) n=5
    l=266 b= 24 W=+16.906 (frac=+2.02, z=+26.83) v_peak=-3.0 km/s (dv=+1.3, z=+0.43) n=15
    l=228 b= 12 W=+4.264 (frac=-0.31, z=-3.63) v_peak=+19.7 km/s (dv=+14.2, z=+1.73) n=15
    l= 12 b= -4 W=+14.074 (frac=+1.54, z=+6.51) v_peak=+7.6 km/s (dv=+4.6, z=+1.53) n=5
    l=250 b= 24 W=+0.189 (frac=-0.96, z=-4.68) v_peak=-6.1 km/s (dv=-0.6, z=-0.20) n=20
    l=242 b= 22 W=+0.833 (frac=-0.86, z=-5.71) v_peak=-3.0 km/s (dv=-1.5, z=-0.50) n=20
    l=254 b=  8 W=

## 5. Save reduced data

Serialize the frequency-switched spectra, velocity grid, cell metrics,
QA flags, and per-cell LO dump counts to a single `.npz` file.  Downstream
notebooks (`02a_scan_diagnostics`, `02b_survey_results`) load this file
instead of reprocessing raw dumps.

In [43]:
REDUCED_PATH = Path('../reduced_survey.npz')

# Build arrays for serialization
cell_keys_arr = np.array(list(cell_results_combined.keys()), dtype=int)  # (N, 2)
R_stack = np.array([cell_results_combined[(gl, gb)]['R_overlap']
                     for gl, gb in cell_keys_arr], dtype=float)           # (N, n_ch)
n_pairs_arr = np.array([cell_results_combined[(gl, gb)]['n_pairs']
                          for gl, gb in cell_keys_arr], dtype=int)

# Per-cell LO dump counts (for manifest generation downstream)
lo_counts = {}
for r in records:
    if r.get('gl') is None or r['noise_on']:
        continue
    key = (r['gl'], r['gb'])
    if key not in lo_counts:
        lo_counts[key] = {'n_1420': 0, 'n_1421': 0}
    if r['lo_mhz'] == 1420.0:
        lo_counts[key]['n_1420'] += 1
    elif r['lo_mhz'] == 1421.0:
        lo_counts[key]['n_1421'] += 1

n_1420_arr = np.array([lo_counts.get((gl, gb), {}).get('n_1420', 0)
                        for gl, gb in cell_keys_arr], dtype=int)
n_1421_arr = np.array([lo_counts.get((gl, gb), {}).get('n_1421', 0)
                        for gl, gb in cell_keys_arr], dtype=int)

# QA metrics and flags
metrics_keys = ['W', 'peak_R', 'peak_v', 'peak_prom', 'snr', 'noise_rms',
                'W_frac_resid', 'W_z', 'peak_v_resid', 'peak_v_z',
                'neighbor_count']
metrics_lookup = {(c['gl'], c['gb']): c for c in neighbor_cells}
metrics_arrays = {}
for mk in metrics_keys:
    metrics_arrays[mk] = np.array(
        [metrics_lookup.get((gl, gb), {}).get(mk, np.nan)
         for gl, gb in cell_keys_arr], dtype=float,
    )

W_flag = np.array([metrics_lookup.get((gl, gb), {}).get('W_flag', False)
                    for gl, gb in cell_keys_arr], dtype=bool)
peak_v_flag = np.array([metrics_lookup.get((gl, gb), {}).get('peak_v_flag', False)
                         for gl, gb in cell_keys_arr], dtype=bool)

np.savez_compressed(
    REDUCED_PATH,
    cell_keys=cell_keys_arr,
    R_stack=R_stack,
    v_lsr=v_lsr_overlap,
    dv_kms=dv_kms,
    n_pairs=n_pairs_arr,
    n_1420=n_1420_arr,
    n_1421=n_1421_arr,
    W_flag=W_flag,
    peak_v_flag=peak_v_flag,
    **metrics_arrays,
)
print(f'Saved reduced data: {REDUCED_PATH}')
print(f'  {len(cell_keys_arr)} cells, {R_stack.shape[1]} channels')
print(f'  QA flags: W={W_flag.sum()}, peak_v={peak_v_flag.sum()}')

Saved reduced data: ../reduced_survey.npz
  1636 cells, 418 channels
  QA flags: W=16, peak_v=4


## 6. Summary

In [44]:
all_times = [r['time'] for r in records]
t0 = dt.datetime.fromtimestamp(min(all_times), tz=dt.UTC)
t1 = dt.datetime.fromtimestamp(max(all_times), tz=dt.UTC)

n_sci = sum(1 for r in records if not r['noise_on'])

NBLOCKS = 1025
NSAMPLES = 32768
T_INT = NBLOCKS * NSAMPLES / SAMPLE_RATE_HZ  # 13.107 s per dump
total_int_s = n_sci * T_INT

gl_vals = sorted(set(r['gl'] for r in records if r['gl'] is not None))
gb_vals = sorted(set(r['gb'] for r in records if r['gb'] is not None))

print(f'Scan grid:  {len(gb_vals)} x {len(gl_vals)} = {len(cell_results_combined)} cells')
print(f'LO freqs:   {lo_unique} MHz')
print(f'Start:      {t0:%Y-%m-%d %H:%M:%S UTC}')
print(f'End:        {t1:%Y-%m-%d %H:%M:%S UTC}')
print(f'Integration: {total_int_s/3600:.1f} hr ({n_sci} science dumps x {T_INT:.1f} s)')
print(f'Channels:   {NFFT} (dnu={SAMPLE_RATE_HZ/NFFT/1e3:.2f} kHz)')
print(f'Neighbor QA: {len(neighbor_cells)} cells analyzed')
print(f'  W flags: {neighbor_w_flag_count}, peak_v flags: {neighbor_peak_flag_count}')
print(f'Reduced data: {REDUCED_PATH}')

Scan grid:  34 x 235 = 1636 cells
LO freqs:   [1420.0, 1421.0] MHz
Start:      2026-04-16 09:59:51 UTC
End:        2026-04-24 09:12:52 UTC
Integration: 50.2 hr (13765 science dumps x 13.1 s)
Channels:   1024 (dnu=2.50 kHz)
Neighbor QA: 1636 cells analyzed
  W flags: 16, peak_v flags: 4
Reduced data: ../reduced_survey.npz
